# K-Nearest Neighbors (KNN) Classification

## Definition
KNN is a **lazy, instance-based** supervised learning algorithm. It makes predictions by finding the **K closest training samples** in feature space and returning the **majority class** among them. There is no explicit training phase — the model memorises the entire training set.

## Core Concept

### Distance Metrics
| Metric | Formula |
|--------|---------|
| **Euclidean** (default) | $d = \sqrt{\sum_i (x_i - x'_i)^2}$ |
| **Manhattan** | $d = \sum_i |x_i - x'_i|$ |
| **Minkowski** | Generalisation of the above |

### Prediction
For a new sample $x$:
1. Compute distance to every training sample.
2. Select the **K nearest** neighbours.
3. Assign the **majority class** among them.

### Choosing K
| Small K | Large K |
|---------|--------|
| Low bias, high variance (overfitting) | High bias, low variance (underfitting) |
| Sensitive to noise | Smoother decision boundary |

## Applications
- **Recommendation systems** — Finding similar users/items
- **Image classification** — Pixel-similarity based recognition
- **Anomaly detection** — Distance-based outlier identification
- **Medical diagnosis** — Pattern similarity across patient records

---

## Why Wine Dataset?
> `load_wine()` has **3 classes** and **13 chemical features** (alcohol, malic acid, flavanoids, etc.) from wine cultivars in Italy. The features differ considerably in scale, highlighting why **StandardScaler is essential for KNN** (distance-based algorithms are scale-sensitive). It is a compact, well-separated dataset that clearly shows the effect of K and scaling.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
data = load_wine()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['class'] = df['target'].map(dict(enumerate(data.target_names)))
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
X = df[data.feature_names]; y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
# Find optimal K using cross-validation
k_range = range(1, 21)
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_s, y_train, cv=5).mean()
             for k in k_range]

best_k = k_range[np.argmax(cv_scores)]
print(f'Best K = {best_k}  (CV Accuracy = {max(cv_scores):.4f})')

plt.figure(figsize=(8,4))
plt.plot(k_range, cv_scores, marker='o', color='darkorange')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K'); plt.ylabel('CV Accuracy')
plt.title('KNN — Choosing Optimal K'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(cmap='Oranges')
plt.title(f'KNN (K={best_k}) — Confusion Matrix'); plt.tight_layout(); plt.show()